In [79]:
import qr
import importlib
importlib.reload(qr)

<module 'qr' from '/home/egorobi/Courses/VLCS/inbedded/qr.py'>

In [80]:
import math
import numpy as np
import segno
import cv2
import moderngl

In [81]:
def create_qr(content):
    qr = segno.make_qr(content)
    matrix = qr.matrix
    arr = np.array(matrix, dtype=int)

    qr.save("output/qr.png", scale=10)

In [82]:
qcd = cv2.QRCodeDetector()

def read_qr(img):
    ret, decoded_info, points, straight_qrcode = qcd.detectAndDecodeMulti(img)

    return decoded_info

In [83]:
msg = "hello world"

for word in msg.split(" "):
    create_qr(word)
    print(read_qr(cv2.imread("output/qr.png")))

('hello',)
('world',)


In [84]:
create_qr("shaders")

bg_img = cv2.imread('wide_cat.jpg')
fg_img = cv2.imread('output/qr.png', cv2.IMREAD_UNCHANGED)

h, w, _ = bg_img.shape
h_fg, w_fg = fg_img.shape

bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGBA)
fg_img = cv2.cvtColor(fg_img, cv2.COLOR_BGRA2RGBA)

ctx = moderngl.create_standalone_context()

bg_tex = ctx.texture((w, h), 4, bg_img.tobytes())
fg_tex = ctx.texture((w_fg, h_fg), 4, fg_img.tobytes())
out_tex = ctx.texture((w, h), 4)

In [85]:
compute_shader = ctx.compute_shader(
    """
    #version 430

    layout(local_size_x = 16, local_size_y = 16) in;

    layout(rgba8, binding = 0) uniform image2D bgImage;
    layout(rgba8, binding = 1) uniform image2D fgImage;
    layout(rgba8, binding = 2) uniform image2D outImage;

    uniform bool u_phase;

    void main() {
        ivec2 coords = ivec2(gl_GlobalInvocationID.xy);
        
        ivec2 size = imageSize(outImage);
        ivec2 fgSize = imageSize(fgImage);
        if (coords.x >= size.x || coords.y >= size.y) {
            return;
        }

        vec4 bg = imageLoad(bgImage, coords);
        vec4 fg = imageLoad(fgImage, coords);

        vec3 finalColor;
        if (fg.rgb[0] > 0.5 || coords.x >= fgSize.x || coords.y >= fgSize.y){
            finalColor = bg.rgb;
        }
        else if (u_phase){
            finalColor = vec3(1.0);
        }
        else {
            finalColor = vec3(0.0);
        }

        imageStore(outImage, coords, vec4(finalColor, 1.0));
    }
"""
)

In [86]:
bg_tex.bind_to_image(0, read=True, write=False)
fg_tex.bind_to_image(1, read=True, write=False)
out_tex.bind_to_image(2, read=False, write=True)

group_x = math.ceil(w / 16)
group_y = math.ceil(h / 16)

for i in range(2):
    compute_shader['u_phase'].value = i % 2
    compute_shader.run(group_x, group_y, 1)

    result_data = out_tex.read()
    result_img = np.frombuffer(result_data, dtype=np.uint8).reshape((h, w, 4))

    result_img = cv2.cvtColor(result_img, cv2.COLOR_RGBA2BGR)
    cv2.imwrite(f"output/overlay_{i}.png", result_img)

    cv2.imshow('Compute Shader Result', result_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [87]:
img0 = cv2.imread("output/overlay_0.png")
img1 = cv2.imread("output/overlay_1.png")

dif = cv2.absdiff(img0, img1)
dif = 255 - dif

print(read_qr(dif))

('shaders',)
